In [1]:
!pip install zss -q
!pip install antlr4-python3-runtime==4.11 -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
secret_label = "GITHUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [3]:
import os

token = secret_value

repo_url = f"https://{token}@github.com/smiling-demon/llm-symbolic-ode-reasoning-dev.git"
repo_dir = "llm-symbolic-ode-reasoning-dev"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)

!pwd

Cloning into 'llm-symbolic-ode-reasoning-dev'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 128 (delta 50), reused 107 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.50 MiB | 10.35 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/kaggle/working/llm-symbolic-ode-reasoning-dev


In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer

from llm import LLM
from methods import train_reasoning_bank, solve_with_reasoning_bank
from metrics import evaluate_predictions

In [5]:
df_train = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/train.xlsx")
df_test = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/test.xlsx")


equations_train = df_train['equation'].tolist()
solutions_train = df_train['solution'].tolist()

equations_test = df_test['equation'].tolist()
solutions_test = df_test['solution'].tolist()

In [6]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
%%time

batch_size = 10
predictions = []

bank = train_reasoning_bank(llm,
                            embedding_model,
                            equations_train,
                            solutions_train,
                            batch_size=batch_size,
                            max_new_tokens=1024)

predictions = solve_with_reasoning_bank(bank,
                                        llm,
                                        embedding_model,
                                        equations_test,
                                        batch_size=batch_size,
                                        max_new_tokens=1024)

CPU times: user 41min 25s, sys: 59.1 s, total: 42min 24s
Wall time: 42min 23s


In [8]:
evaluation = evaluate_predictions(predictions, solutions_test)
for key in evaluation:
    evaluation[key] = [evaluation[key]]
pd.DataFrame.from_dict(evaluation)

,bleu,ast_error_size,ast_tree_similarity,exact_match
0,0.561087,0.529929,0.350858,0.41
